# DeepWalk — Online Learning of Social Representations (2014)

_Papers · Graph Neural Networks_

**Generate truncated random walks from each node, treat each walk as a sentence, and run word2vec's skip-gram on them — nodes that share neighborhoods get similar vectors.**

---

This notebook builds DeepWalk up one piece at a time: first the skip-gram pair extractor on a hand-checkable walk, then a toy two-community graph, then truncated random walks feeding skip-gram with negative sampling, and finally an ablation that probes the learned embeddings against the raw adjacency rows. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Extract skip-gram pairs from one walk

DeepWalk treats each random walk as a "sentence" and runs word2vec's skip-gram on it. For each position (the *center*) we emit `(center, context)` pairs for every node within a symmetric window of half-width `w`. We check the extractor on a hand-worked length-5 walk at `w=2`, which should produce exactly 15 pairs.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

random.seed(0)
torch.manual_seed(0)

# Worked example: one length-5 walk -> skip-gram (center, context) pairs at window w=2.
def skipgram_pairs(walk, w):
    pairs = []
    for i, c in enumerate(walk):
        for j in range(max(0, i - w), min(len(walk), i + w + 1)):
            if j != i:
                pairs.append((c, walk[j]))
    return pairs

ex = skipgram_pairs([3, 1, 4, 1, 5], w=2)
print("walk (3,1,4,1,5), w=2 ->", ex)
print("number of pairs =", len(ex))     # 15, matching the worked example by hand

### Step 2 — Build a toy graph and the random-walk sampler

We construct a 12-node graph with two dense communities (0–5 and 6–11) joined by a single bridge edge (0,6). DeepWalk's walk rule is simple: from the current node, step to a *uniformly random* neighbor (the unbiased walk). We generate `gamma` walks starting from every node, which gives skip-gram plenty of co-occurrence evidence within each community.

In [ ]:
# A toy graph: two dense communities joined by a single bridge.
# Community A = nodes 0..5, Community B = nodes 6..11. Each has a hub wired to its members plus a ring;
# one bridge edge (0,6) connects the communities. Same graph used across the GNN paper lessons.
def make_graph():
    edges = set()
    def add(u, v):
        edges.add((min(u, v), max(u, v)))
    for comm in ([0,1,2,3,4,5], [6,7,8,9,10,11]):
        hub = comm[0]
        for u in comm[1:]:
            add(hub, u)                       # hub connects to everyone in its community
        for i in range(1, len(comm)):         # plus a ring so the community is dense
            add(comm[i], comm[(i % (len(comm)-1)) + 1])
    add(0, 6)                                  # the single bridge between the two hubs
    N = 12
    adj = [[] for _ in range(N)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    return N, [sorted(set(a)) for a in adj]

N, adj = make_graph()

# Truncated random walk: each step picks a UNIFORMLY-random neighbor (unbiased = DeepWalk).
def random_walk(start, length):
    path = [start]
    cur = start
    for _ in range(length - 1):
        cur = random.choice(adj[cur])          # uniform over neighbors -- the whole walk rule
        path.append(cur)
    return path

def sample_walks(gamma=20, length=20):
    walks = []
    for _ in range(gamma):                     # gamma walks started from EVERY node
        order = list(range(N))
        random.shuffle(order)
        for start in order:
            walks.append(random_walk(start, length))
    return walks

### Step 3 — Learn node embeddings with skip-gram + negative sampling

Now we train the embeddings. For each `(center, context)` pair we pull the center toward its true context (the positive term) and push it away from a handful of random nodes (the negatives) — this is skip-gram with negative sampling, the cheap approximation to the full softmax. After training, nodes that share neighborhoods end up with similar 2-D vectors.

In [ ]:
# Skip-gram with negative sampling over the walks (Eqn. 2).
def train_embeddings(walks, dim=2, window=2, negatives=5, epochs=120, lr=0.05):
    torch.manual_seed(0)
    emb = nn.Embedding(N, dim)
    ctx = nn.Embedding(N, dim)
    nn.init.normal_(emb.weight, std=0.1)
    nn.init.normal_(ctx.weight, std=0.1)
    opt = torch.optim.Adam(list(emb.parameters()) + list(ctx.parameters()), lr=lr)

    pairs = []
    for wlk in walks:
        pairs.extend(skipgram_pairs(wlk, window))   # SAME extractor as the worked example
    pairs = torch.tensor(pairs)

    for _ in range(epochs):
        perm = pairs[torch.randperm(len(pairs))]
        centers, contexts = perm[:, 0], perm[:, 1]
        negs = torch.randint(0, N, (len(perm), negatives))
        opt.zero_grad()
        ce, co = emb(centers), ctx(contexts)
        pos = F.logsigmoid((ce * co).sum(1))                              # pull center -> true context
        neg = F.logsigmoid(-(emb(centers).unsqueeze(1) * ctx(negs)).sum(2)).sum(1)  # push from negatives
        loss = -(pos + neg).mean()
        loss.backward()
        opt.step()
    return emb.weight.detach()

Z = train_embeddings(sample_walks())           # 2-D node embeddings

### Step 4 — Probe the embeddings vs the raw adjacency rows

Finally, the ablation. We fit a logistic-regression probe from just a few labeled nodes, once on the learned 2-D embeddings and once on the raw adjacency rows, then test on the rest. The embedding probe should recover the community split far better — two same-community nodes can share no edge (so their adjacency rows look unrelated) yet still co-occur in walks, which pulls their embeddings together.

In [ ]:
# The ablation: a logistic-regression probe on the EMBEDDINGS vs the raw ADJACENCY rows.
from sklearn.linear_model import LogisticRegression
import numpy as np

labels = np.array([0]*6 + [1]*6)               # community A vs B
train_idx = [0, 1, 6, 7]                        # a few labeled nodes (2 per community)
test_idx  = [2, 3, 4, 5, 8, 9, 10, 11]

# adjacency rows: length-N 0/1 vector of who each node connects to (the sparse baseline)
A = np.zeros((N, N))
for u in range(N):
    for v in adj[u]:
        A[u, v] = 1.0

def probe(X):
    clf = LogisticRegression(max_iter=1000).fit(X[train_idx], labels[train_idx])
    return (clf.predict(X[test_idx]) == labels[test_idx]).mean()

acc_emb = probe(Z.numpy())
acc_adj = probe(A)
print(f"logistic probe accuracy:  DeepWalk 2-D embedding = {acc_emb:.2f}   raw adjacency rows = {acc_adj:.2f}")

# Our small run (NOT the paper's numbers): the length-5 walk gives 15 skip-gram pairs (matches the worked
# example); a logistic probe on the learned 2-D embeddings recovers the community split (~1.00) while the
# same probe on the sparse adjacency rows is weaker -- because two same-community nodes can share no edge,
# so their adjacency rows look unrelated, but their random walks co-occur and pull their embeddings together.

## Visualize the data & results

_Do DeepWalk's random-walk + skip-gram embeddings recover the community split — and does a logistic probe read them more easily than the raw adjacency rows (the ablation)?_

### Step 1 — Rebuild the graph, walks, and embedding trainer

This visualization cell is self-contained. It re-creates the toy two-community graph, the uniform random walk, the skip-gram pair extractor, and the negative-sampling trainer exactly as above, then learns 2-D embeddings — so the probe comparison can run on its own.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np
from sklearn.linear_model import LogisticRegression

random.seed(0)
torch.manual_seed(0)

# Toy graph: two dense communities (0-5, 6-11) joined by a single bridge edge.
def make_graph():
    edges = set()
    def add(u, v):
        edges.add((min(u, v), max(u, v)))
    for comm in ([0,1,2,3,4,5], [6,7,8,9,10,11]):
        hub = comm[0]
        for u in comm[1:]:
            add(hub, u)
        for i in range(1, len(comm)):
            add(comm[i], comm[(i % (len(comm)-1)) + 1])
    add(0, 6)
    adj = [[] for _ in range(12)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    return [sorted(set(a)) for a in adj]

adj = make_graph()
N = 12

def walk(s, L):                                  # truncated UNIFORM random walk (DeepWalk)
    p = [s]
    cur = s
    for _ in range(L-1):
        cur = random.choice(adj[cur])
        p.append(cur)
    return p

def pairs_of(wlk, w):                             # skip-gram pairs: 15 for (3,1,4,1,5),w=2
    return [(c, wlk[j]) for i, c in enumerate(wlk)
            for j in range(max(0, i-w), min(len(wlk), i+w+1)) if j != i]

def deepwalk(dim=2, w=2, neg=5, gamma=20, L=20, epochs=120):
    torch.manual_seed(0)
    walks = [walk(s, L) for _ in range(gamma) for s in range(N)]
    pairs = torch.tensor([p for wl in walks for p in pairs_of(wl, w)])
    emb = nn.Embedding(N, dim)
    ctx = nn.Embedding(N, dim)
    nn.init.normal_(emb.weight, std=0.1)
    nn.init.normal_(ctx.weight, std=0.1)
    opt = torch.optim.Adam(list(emb.parameters()) + list(ctx.parameters()), lr=0.05)
    for _ in range(epochs):
        pr = pairs[torch.randperm(len(pairs))]
        ce, co = pr[:, 0], pr[:, 1]
        ng = torch.randint(0, N, (len(pr), neg))
        opt.zero_grad()
        pos = F.logsigmoid((emb(ce) * ctx(co)).sum(1))
        nge = F.logsigmoid(-(emb(ce).unsqueeze(1) * ctx(ng)).sum(2)).sum(1)
        (-(pos + nge).mean()).backward()
        opt.step()
    return emb.weight.detach().numpy()

Z = deepwalk()

### Step 2 — Run the probe on both representations

With embeddings in hand, we fit the same logistic-regression probe on the 2-D embeddings and on the raw adjacency rows, training on a few labeled nodes and testing on the rest. The embedding probe (~1.00) should clearly beat the adjacency probe (~0.62) — the quantitative read-out of DeepWalk's contribution.

In [ ]:
labels = np.array([0]*6 + [1]*6)
tr = [0, 1, 6, 7]
te = [2, 3, 4, 5, 8, 9, 10, 11]

A = np.zeros((N, N))
for u in range(N):
    for v in adj[u]:
        A[u, v] = 1.0

def probe(X):
    c = LogisticRegression(max_iter=1000).fit(X[tr], labels[tr])
    return (c.predict(X[te]) == labels[te]).mean()

print("probe acc  embedding =", round(probe(Z), 2), " adjacency =", round(probe(A), 2))
# Embedding ~1.00 vs adjacency ~0.62. Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Recount the skip-gram pairs. A random walk produced the sequence $W=(2,\,7,\,2,\,9)$ (length
            4), and you use window half-width $w=1$. List the (center, context) pairs for every center and give
            the total count.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Center pos 0 (node 2): window reaches pos 1 only. Pair $(2,7)$. — _Half-width $w=1$, left edge clipped, so only the immediate right neighbor is context._
- Center pos 1 (node 7): window covers pos 0 and 2. Pairs $(7,2),(7,2)$. — _Both sides are within $1$; node 2 appears at both positions 0 and 2._
- Center pos 2 (node 2): window covers pos 1 and 3. Pairs $(2,7),(2,9)$. — _Same rule one step further along._
- Center pos 3 (node 9): window reaches pos 2 only. Pair $(9,2)$. — _Right edge clipped._

**Answer:** Pairs: $(2,7);\ (7,2),(7,2);\ (2,7),(2,9);\ (9,2)$ &mdash; 6 pairs total. Node $2$
                 appears twice in the short walk, so it co-occurs with both $7$ and $9$; that repeated, frequent
                 node is exactly the power-law (hub) behavior that makes the word model appropriate.

</details>

**Problem 2.** Why does the paper relax the ordered language-modeling objective (Eqn. 1) to the symmetric-window
            skip-gram objective (Eqn. 2)? Give the two reasons and the assumption that makes Eqn. 2 tractable.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Eqn. 1 conditions on the growing prefix $(\Phi(v_1),\dots,\Phi(v_{i-1}))$. — _As the walk lengthens the conditioning context grows without bound — awkward and order-dependent._
- Flip to predicting the surrounding nodes from the center, over a fixed symmetric window $[i-w,i+w]$, dropping order. — _We only care that nodes co-occur near each other, not their order — this is skip-gram (Eqn. 2)._
- Assume the context nodes are conditionally independent given the center. — _The joint $\Pr(\text{context}\mid\Phi(v_i))$ factorizes into a product of per-node terms, so training reduces to per-pair updates._

**Answer:** Two reasons: (1) the ordered prefix in Eqn. 1 grows unboundedly as the walk extends, and
                 (2) order does not matter for capturing co-occurrence &mdash; so the paper predicts the
                 symmetric window of surrounding nodes from the center instead (Eqn. 2). The
                 conditional-independence assumption then factorizes the joint context probability into a
                 product of one-node-at-a-time terms, turning the log-objective into a sum of cheap per-pair
                 (center, context) updates &mdash; computed with hierarchical softmax in the paper, negative
                 sampling in our code.

</details>

**Problem 3.** Ablation: embedding vs raw adjacency. On a 2-community graph you train DeepWalk embeddings,
            then fit a logistic-regression probe on (a) the 2-D embeddings and (b) the length-$N$ adjacency
            rows, each from a few labeled nodes. Predict which probe recovers the community split better, and
            explain via two same-community nodes that share no direct neighbor.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Adjacency row of a node is $1$ on its direct neighbors, $0$ elsewhere. — _Two same-community nodes with no shared neighbor have adjacency rows that overlap in zero positions — they look dissimilar to the probe._
- Random walks from both nodes wander through the same dense community, so they co-occur with overlapping node sets. — _Skip-gram pulls their embeddings together because they share context — community structure is encoded in the dense vector._
- Fit the same logistic probe on each representation; change only the input features. — _Isolating the representation attributes any accuracy gap to the embedding, not the classifier._

**Answer:** The DeepWalk embedding probe wins. Two same-community nodes that share no direct neighbor
                 have disjoint adjacency rows (overlap zero), so a probe reading adjacency cannot tell they
                 belong together. But random walks from both nodes traverse the same dense community and therefore
                 co-occur with an overlapping set of nodes, so skip-gram pulls their embeddings close. The
                 dense walk-embedding encodes \"same neighborhood\" even without a shared edge, which is exactly
                 why a plain logistic regression separates the communities far more accurately on the embedding
                 than on the raw adjacency. The CODEVIZ panel measures this accuracy gap on a toy graph &mdash; a
                 direct read-out of DeepWalk's contribution.

</details>